[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haohanchen/POLI3148_2026Spring_TextAnalysis/blob/main/lecture/session_2_wordfreq_topics/session_2_wordfreq_topics.ipynb)

# Session 2: Word Frequencies, Word Clouds, and Topic Modeling

**POLI3148 -- Data Science in Politics and Public Administration (HKU)**

---

In this session, we move from understanding text *structure* (tokenization, NER) to understanding text *content*. We will use word-based methods to explore what journalists ask and what MoFA spokespeople say at China's Ministry of Foreign Affairs press conferences.

**What we will cover:**

1. Text pre-processing pipeline (tokenization, lemmatization, stopword removal)
2. Bag-of-Words (BoW) representation
3. Word frequencies and word clouds
4. N-grams (bigrams)
5. TF-IDF weighting
6. Topic modeling with Latent Dirichlet Allocation (LDA)

**Data:** MoFA Press Conference Corpus (2002--2025), ~35,000 Q&A exchanges.

## Setup

In [ ]:
!pip install wordcloud pyLDAvis spacy -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis
import pyLDAvis.lda_model
import warnings
warnings.filterwarnings("ignore")

# Plotting defaults
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

In [ ]:
# Load the MoFA Press Conference corpus
df = pd.read_excel("https://github.com/haohanchen/POLI3148_2026Spring_TextAnalysis/raw/main/data/CMFA_PressCon_v6.xlsx")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Take a sample of 10,000 Q & As (for faster computation).
# Comment it out if you want to use the full dataset (but it will be much slower).
df = df.sample(n=10000, random_state=42).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Part 1: Text Pre-processing Pipeline

Before we can count words meaningfully, we need to clean and normalize the text. Our pipeline:

1. **Tokenize** -- split text into individual words (tokens)
2. **Lemmatize** -- reduce words to their base form ("countries" -> "country", "said" -> "say")
3. **Remove noise** -- drop stopwords ("the", "is", "and"), punctuation, numbers, and very short tokens

We will use **spaCy** to do all of this in one pass.

In [ ]:
# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Domain-specific stopwords: extremely common in MoFA press conferences
# but carry little analytical value
domain_stopwords = {"china", "chinese", "say", "comment", "question", "report",
                    "country", "state", "government", "minister", "foreign",
                    "people", "spokesperson", "medium", "press", "conference",
                    "regard", "side", "note", "relate", "concern", "issue"}

def preprocess_text(text):
    """Tokenize, lemmatize, and clean a single text string.
    Returns a list of cleaned lemma strings."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop       # remove stopwords
        and not token.is_punct     # remove punctuation
        and not token.like_num     # remove numbers
        and len(token.lemma_) >= 2 # remove very short tokens
        and token.lemma_.lower() not in domain_stopwords  # remove domain stopwords
    ]
    return tokens

# NOTE: If domain stopwords still appear in word frequency plots,
# make sure to re-run ALL cells from this point onward (the tokens
# need to be regenerated with the updated stopword list).


In [ ]:
# Demo: preprocess one question
sample_text = df["question"].dropna().iloc[0]
print("Original:", sample_text[:300], "...")
print()
print("Tokens:", preprocess_text(sample_text))

In [ ]:
# Process ALL questions using nlp.pipe() for efficiency
print(f"Processing {len(df)} questions... this may take a few minutes.")

texts = df["question"].dropna().astype(str).tolist()
processed_docs = list(nlp.pipe(texts, disable=["ner"], batch_size=200))

# Extract tokens using the same logic as preprocess_text()
df_processed = df.dropna(subset=["question"]).copy()
df_processed["tokens"] = [
    [t.lemma_.lower() for t in doc
     if not t.is_stop
     and not t.is_punct
     and not t.like_num
     and len(t.lemma_) >= 2
     and t.lemma_.lower() not in domain_stopwords]  # remove domain stopwords
    for doc in processed_docs
]

print("Done!")
print(f"Total documents processed: {len(df_processed)}")
print(f"Average tokens per question: {df_processed['tokens'].apply(len).mean():.1f}")
df_processed[["id", "question", "tokens"]].head(3)


## Part 2: Bag-of-Words (BoW) Model

The **Bag-of-Words** model represents each document as a vector of word counts, completely ignoring word order. It is the simplest and most foundational text representation.

**Example:** Consider two sentences:
- "She loves pizza" -> {she: 1, loves: 1, pizza: 1}
- "Pizza loves she" -> {she: 1, loves: 1, pizza: 1}

Both get the *same* representation -- the "bag" discards word order. This is a major limitation, but BoW is surprisingly effective for many tasks.

We use scikit-learn's `CountVectorizer` to build the **document-term matrix (DTM)**: a matrix where rows are documents and columns are words, and each cell contains the word count.

In [ ]:
# Since we already tokenized, we pass tokens directly with a dummy tokenizer
def identity_tokenizer(x):
    return x

vectorizer = CountVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=lambda x: x,  # no additional preprocessing
    lowercase=False,            # already lowercased
    min_df=10,                  # ignore words appearing in fewer than 10 docs
    max_df=0.5,                 # ignore words appearing in more than 50% of docs
)

dtm = vectorizer.fit_transform(df_processed["tokens"])
vocab = vectorizer.get_feature_names_out()

print(f"Document-term matrix shape: {dtm.shape}")
print(f"  -> {dtm.shape[0]} documents, {dtm.shape[1]} unique terms")
print(f"\nFirst 20 vocabulary terms (alphabetical): {list(vocab[:20])}")


In [ ]:
# Peek at the BoW matrix: first 5 documents, first 20 terms
slice_df = pd.DataFrame(
    dtm[:5, :20].toarray(),
    columns=vocab[:20],
    index=[f"Doc {i}" for i in range(5)]
)
slice_df

## Part 3: Word Frequencies

The simplest thing we can do with a document-term matrix: sum each column to get the total frequency of every word across all documents.

In [ ]:
# Total word frequencies across all documents
word_freq = pd.Series(dtm.sum(axis=0).A1, index=vocab).sort_values(ascending=False)
print(f"Total unique terms: {len(word_freq)}")
print(f"\nTop 10 words:")
word_freq.head(10)

In [ ]:
# Plot top 30 most frequent words
fig, ax = plt.subplots(figsize=(8, 7))
top30 = word_freq.head(30)
top30.iloc[::-1].plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Frequency")
ax.set_title("Top 30 Most Frequent Words in MoFA Questions")
plt.tight_layout()
plt.show()

**Discussion:** The most frequent words likely include "china", "chinese", "say", "question", "comment" -- words that appear in almost every press conference question but do not tell us much about the *topic*. These are sometimes called **domain-specific stopwords**. Let's remove them and re-examine.

In [ ]:
# Domain stopwords already removed during pre-processing.
# Plot the top 30 words directly.

fig, ax = plt.subplots(figsize=(8, 7))
top30 = word_freq.head(30)
top30.iloc[::-1].plot.barh(ax=ax, color="darkorange")
ax.set_xlabel("Frequency")
ax.set_title("Top 30 Most Frequent Words in MoFA Questions")
plt.tight_layout()
plt.show()


## Part 4: Word Clouds

Word clouds are a popular (if informal) way to visualize word frequencies. The size of each word is proportional to its frequency.

In [ ]:
# Create word frequency dictionary (using filtered frequencies)
freq_dict = word_freq.to_dict()

# Basic word cloud
wc = WordCloud(width=1000, height=600, background_color="white",
               colormap="viridis", random_state=42, max_words=150)
wc.generate_from_frequencies(freq_dict)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud -- MoFA Press Conference Questions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Comparative word clouds: all major spokespeople
def get_tokens_for_spokesperson(name):
    """Get word frequencies for questions directed at a given spokesperson."""
    mask = df_processed["spokesperson"] == name
    tokens_list = df_processed.loc[mask, "tokens"].tolist()
    all_tokens = [t for doc in tokens_list for t in doc]
    freq = pd.Series(all_tokens).value_counts()
    freq = freq.drop(labels=[w for w in domain_stopwords if w in freq.index], errors="ignore")
    return freq.to_dict()

# Get top spokespeople (by number of Q&As)
top_spokespersons = df_processed["spokesperson"].value_counts().head(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for ax, name in zip(axes, top_spokespersons):
    freq = get_tokens_for_spokesperson(name)
    if len(freq) > 0:
        wc = WordCloud(width=600, height=400, background_color="white",
                       colormap="viridis", random_state=42, max_words=80)
        wc.generate_from_frequencies(freq)
        ax.imshow(wc, interpolation="bilinear")
    n_qa = (df_processed["spokesperson"] == name).sum()
    ax.set_title(f"{name} (n={n_qa})", fontsize=11)
    ax.axis("off")

plt.suptitle("Word Clouds by Spokesperson -- What Topics Did Journalists Ask About?", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## Part 5: N-grams

Single words (unigrams) miss important multi-word expressions. "South China Sea" as three separate words -- "south", "china", "sea" -- loses the meaning of the phrase. **N-grams** are contiguous sequences of N words:

- **Unigram** (N=1): "south", "china", "sea"
- **Bigram** (N=2): "south china", "china sea"
- **Trigram** (N=3): "south china sea"

Let's look at frequent bigrams in our corpus.

In [ ]:
# Build a vectorizer that captures both unigrams and bigrams
vec_ngram = CountVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=lambda x: x,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=5,
)

dtm_ngram = vec_ngram.fit_transform(df_processed["tokens"])
vocab_ngram = vec_ngram.get_feature_names_out()

print(f"With bigrams: {dtm_ngram.shape[1]} features (vs. {dtm.shape[1]} unigrams only)")

# Show a slice of the bigram DTM -- pick some bigram columns to illustrate
bigram_cols = [v for v in vocab_ngram if " " in v][:10]
bigram_indices = [list(vocab_ngram).index(b) for b in bigram_cols]
slice_bigram = pd.DataFrame(
    dtm_ngram[:5, bigram_indices].toarray(),
    columns=bigram_cols,
    index=[f"Doc {i}" for i in range(5)]
)
print("\nSample bigram columns from the DTM (first 5 documents):")
display(slice_bigram)

**Discussion:** Bigrams capture meaningful phrases that unigrams cannot: "hong kong", "north korea", "south china", "human right", "united states", etc. These give us a much richer picture of the topics journalists ask about.

## Part 6: TF-IDF

Raw word counts treat all words equally. But a word that appears in *every* document is less informative than one that appears in only a few. **TF-IDF** (Term Frequency -- Inverse Document Frequency) addresses this:

- **TF (Term Frequency):** How often does this word appear in *this* document?
- **IDF (Inverse Document Frequency):** How rare is this word across *all* documents? Computed as log(N / df), where N = total documents and df = number of documents containing the word.
- **TF-IDF = TF x IDF:** Words that are frequent in a document but rare across the corpus get high scores.

This weighting scheme automatically downweights common words and highlights distinctive ones.

In [ ]:
# Build TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=lambda x: x,
    lowercase=False,
    min_df=5,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df_processed["tokens"])
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

# Mean TF-IDF score for each term across all documents
mean_tfidf = pd.Series(
    tfidf_matrix.mean(axis=0).A1, index=tfidf_vocab
).sort_values(ascending=False)

In [ ]:
# Compare the raw count DTM vs. TF-IDF DTM for the same terms
# Pick words that remain in our vocabulary after domain stopword removal
compare_words = ["taiwan", "trade", "nuclear", "visit", "president", "japan",
                 "military", "korea", "sanction", "hong"]
compare_indices = [list(vocab).index(w) for w in compare_words if w in vocab]
compare_words_found = [vocab[i] for i in compare_indices]

tfidf_indices = [list(tfidf_vocab).index(w) for w in compare_words_found]

print("Raw counts (first 5 docs):")
display(pd.DataFrame(
    dtm[:5, compare_indices].toarray(),
    columns=compare_words_found,
    index=[f"Doc {i}" for i in range(5)]
))

print("\nTF-IDF weights (first 5 docs):")
display(pd.DataFrame(
    tfidf_matrix[:5, tfidf_indices].toarray().round(3),
    columns=compare_words_found,
    index=[f"Doc {i}" for i in range(5)]
))

print('\nNotice: common words like "visit" and "president" get lower TF-IDF weights,')
print('while rarer, more specific words like "nuclear" or "sanction" get higher weights when they appear.')


In [ ]:
# Compare: top words by raw frequency vs. top words by mean TF-IDF
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left: raw frequency
top20_raw = word_freq.head(20)
top20_raw.iloc[::-1].plot.barh(ax=axes[0], color="steelblue")
axes[0].set_xlabel("Raw Count")
axes[0].set_title("Top 20 by Raw Frequency")

# Right: mean TF-IDF
top20_tfidf = mean_tfidf.head(20)
top20_tfidf.iloc[::-1].plot.barh(ax=axes[1], color="darkred")
axes[1].set_xlabel("Mean TF-IDF Score")
axes[1].set_title("Top 20 by Mean TF-IDF")

plt.suptitle("Raw Frequency vs. TF-IDF: Which List Is More Informative?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Discussion:** The raw frequency list is dominated by generic, high-frequency words ("china", "say", "question"). The TF-IDF list surfaces words that are both frequent and *distinctive* -- terms that actually help us understand what specific topics the corpus covers. TF-IDF is generally more informative for content analysis.

## Part 7: Topic Modeling (LDA)

**Latent Dirichlet Allocation (LDA)** is an unsupervised method that discovers hidden "topics" in a collection of documents.

**Key intuitions:**
- Each **document** is a mixture of topics. For example, a question might be 60% "trade" and 40% "diplomacy".
- Each **topic** is a mixture of words. For example, the "trade" topic might assign high probability to "tariff", "trade", "export", "deficit".
- LDA discovers these mixtures from the data -- we do not label topics in advance.

**Inputs:**
- A document-term matrix (our BoW matrix)
- K = the number of topics (we choose this)

**Outputs:**
- Topic-word distributions: for each topic, which words are most likely?
- Document-topic distributions: for each document, which topics are most present?

In [ ]:
# Fit LDA with K=15 topics
lda_model = LatentDirichletAllocation(
    n_components=15, # This controls the number of "topics"
    random_state=42,
    max_iter=20,
    learning_method="online",
    n_jobs=-1,
)

lda_model.fit(dtm)
print(f"LDA model fitted with {lda_model.n_components} topics.")

In [ ]:
# Display top 10 words for each topic
def plot_top_words(model, feature_names, n_top_words=10, n_cols=3):
    n_topics = model.n_components
    n_rows = (n_topics + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3 * n_rows))
    axes = axes.flatten()

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:]
        top_words = [feature_names[i] for i in top_indices]
        top_weights = topic[top_indices]

        axes[topic_idx].barh(top_words, top_weights, color="steelblue")
        axes[topic_idx].set_title(f"Topic {topic_idx + 1}", fontsize=11, fontweight="bold")
        axes[topic_idx].tick_params(axis="y", labelsize=9)

    # Hide unused subplots
    for idx in range(n_topics, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle("LDA Topics -- Top 10 Words per Topic", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

plot_top_words(lda_model, vocab, n_top_words=10, n_cols=3)

In [ ]:
# Get document-topic distributions
doc_topics = lda_model.transform(dtm)

# Topic-word distribution: top words for each topic
topic_word_df = pd.DataFrame(
    lda_model.components_,
    columns=vocab,
    index=[f"Topic {i+1}" for i in range(lda_model.n_components)]
)
print("Topic-word distribution (shape):", topic_word_df.shape)
print("Each row = a topic, each column = a word, values = word importance in that topic")
display(topic_word_df.iloc[:3, :8])  # preview

print("\n" + "="*60 + "\n")

# Document-topic distribution: merge with original metadata
doc_topic_df = pd.DataFrame(
    doc_topics,
    columns=[f"Topic {i+1}" for i in range(lda_model.n_components)]
)

# Merge with original dataframe metadata
doc_topic_df = pd.concat([
    df_processed[["id", "spokesperson", "year", "date", "question"]].reset_index(drop=True),
    doc_topic_df
], axis=1)

print("Document-topic distribution (shape):", doc_topic_df.shape)
print("Each row = a document, Topic columns = probability of that topic")

# Show head sorted by Topic 6 (highest first)
print("\nTop documents by Topic 6 probability:")
display(doc_topic_df.sort_values("Topic 6", ascending=False).head(10))


In [ ]:
# Interactive visualization with pyLDAvis
panel = pyLDAvis.lda_model.prepare(lda_model, dtm, vectorizer)
pyLDAvis.display(panel)

In [ ]:
# Document-topic assignments: which topic dominates each question?
doc_topics = lda_model.transform(dtm)

# Show assignments for 5 sample documents
sample_indices = [0, 100, 500, 1000, 2000]
for i in sample_indices:
    dominant_topic = doc_topics[i].argmax() + 1  # 1-indexed
    topic_prob = doc_topics[i].max()
    question_preview = df_processed["question"].iloc[i][:120]
    print(f"Doc {i}: Topic {dominant_topic} ({topic_prob:.2f})")
    print(f"  \"{question_preview}...\"")
    print()

**Discussion:** Look at the top words for each topic. Can you assign a human-readable label to each one? For example, a topic with words like "trade", "tariff", "economic" might be labeled "Trade/Economics". Some topics will be clear; others may be harder to interpret. This is a known challenge with LDA -- the number of topics K and the interpretability of results require judgment and experimentation.

### Experimenting with the Number of Topics (K)

The choice of K is a judgment call. Let's compare K=15 (above) with K=25 to see how more topics changes interpretability.

In [ ]:
# Fit LDA with K=20
lda_20 = LatentDirichletAllocation(
    n_components=20, random_state=42, max_iter=20,
    learning_method="online", n_jobs=-1,
)
lda_20.fit(dtm)
print("LDA with K=20 fitted.")

plot_top_words(lda_20, vocab, n_top_words=8, n_cols=5)

### Labeling Topics and Analyzing by Spokesperson

Now let's put the topic modeling results to use. We will:
1. Label the interpretable topics from our K=15 model based on top words
2. Assign each document its dominant topic
3. Analyze how different spokespeople are associated with different topics

In [ ]:
# Step 1: Print top words for each topic so we can label them
print("K=15 Topic Top Words (for labeling):\n")
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [vocab[i] for i in topic.argsort()[-8:]][::-1]
    print(f"  Topic {topic_idx + 1:2d}: {', '.join(top_words)}")

In [ ]:
# Compute average topic distribution for each spokesperson
# (Instead of assigning a single dominant topic per document,
#  we use the full probability distribution -- more faithful to the model.)

# Add topic probabilities to the dataframe
topic_cols = [f"Topic {i+1}" for i in range(lda_model.n_components)]
doc_topic_probs = pd.DataFrame(doc_topics, columns=topic_cols)

df_topics = pd.concat([
    df_processed[["id", "spokesperson", "year"]].reset_index(drop=True),
    doc_topic_probs
], axis=1)

# Average topic probability for each spokesperson
top_speakers = df_topics["spokesperson"].value_counts().head(10).index.tolist()
speaker_avg = df_topics[df_topics["spokesperson"].isin(top_speakers)].groupby("spokesperson")[topic_cols].mean()
speaker_avg = speaker_avg.loc[top_speakers]  # preserve order by frequency

print("Average topic probability by spokesperson (rows = spokespersons, columns = topics):")
display(speaker_avg.round(3))


In [ ]:
# Visualize spokesperson-topic associations as a heatmap
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    speaker_avg.T,  # transpose: rows = topics, columns = spokespersons
    cmap="YlOrRd",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Average Topic Probability by Spokesperson", fontsize=14)
ax.set_ylabel("Topic")
ax.set_xlabel("Spokesperson")
plt.tight_layout()
plt.show()


---

## Your Turn

### Exercise 1: Comparative Word Clouds by Country

Filter `df_processed` to questions that mention "Japan" vs. questions that mention "US" (use the `q_loc` column -- it contains semicolon-separated location entities, or "-" if none). Generate comparative word clouds for these two subsets.

**Hint:** Use `df_processed[df_processed["q_loc"].str.contains("Japan", na=False)]` to filter rows.

### Exercise 2: Label the Topics

Go back to the topic labeling cell above. Based on the top words for each topic, assign meaningful labels (e.g., "Trade & Economics", "Taiwan/Cross-Strait", "Japan Relations"). Then re-run the spokesperson-topic analysis. Which spokespeople are associated with which topics?

### Vibe Coding Prompt

Try giving this prompt to an AI coding assistant:

> "Modify this notebook to analyze the `answer` column instead of `question`. Are the topics different? Do the spokespeople's word frequencies differ from the journalists' word frequencies?"